# Crop Recommendation System
## Data Preprocessing, EDA & Model Evaluation Notebook

**Dataset:** Crop Recommendation Dataset (N, P, K, temperature, humidity, pH, rainfall → crop label)

**Steps covered:**
1. Load & inspect dataset
2. Exploratory Data Analysis (EDA)
3. Data preprocessing (normalization, encoding)
4. Train/test split
5. Train Random Forest & KNN models
6. Evaluate (accuracy, confusion matrix, classification report)
7. Feature importance
8. Model comparison

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded ✓')

## Step 2: Load Dataset

In [ ]:
df = pd.read_csv('Crop_recommendation.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Data types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nCrops:', sorted(df['label'].unique()))

In [ ]:
df.describe().round(2)

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Crop distribution
fig, ax = plt.subplots(figsize=(12, 5))
df['label'].value_counts().plot(kind='bar', ax=ax, color='#185FA5', edgecolor='white')
ax.set_title('Number of Samples per Crop', fontsize=14)
ax.set_xlabel('Crop'); ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, feat in zip(axes.flatten()[:7], features):
    ax.hist(df[feat], bins=30, color='#1D9E75', edgecolor='white', alpha=0.85)
    ax.set_title(feat, fontsize=12)
    ax.set_xlabel('Value'); ax.set_ylabel('Count')
axes.flatten()[-1].axis('off')
plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots per feature grouped by crop
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, feat in zip(axes.flatten()[:7], features):
    # show top 8 crops for readability
    top8 = df['label'].value_counts().head(8).index
    sub = df[df['label'].isin(top8)]
    sub.boxplot(column=feat, by='label', ax=ax, grid=False)
    ax.set_title(feat); ax.set_xlabel('')
    plt.sca(ax); plt.xticks(rotation=45, ha='right')
axes.flatten()[-1].axis('off')
plt.suptitle('Feature Distribution by Crop (top 8)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[features].corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

## Step 4: Preprocessing — Encoding & Normalization

In [ ]:
# Encode target labels
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])
print('Classes:', list(le.classes_))

X = df[features].values
y = df['label_enc'].values
print(f'\nX shape: {X.shape}, y shape: {y.shape}')

## Step 5: Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training samples:   {X_train.shape[0]}')
print(f'Test samples:       {X_test.shape[0]}')
print(f'Train fraction:     {X_train.shape[0] / len(X):.0%}')

In [ ]:
# Normalize features using StandardScaler
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit ONLY on train
X_test_sc  = scaler.transform(X_test)       # apply same scaler to test

print('Before scaling — X_train mean:', X_train[:,0].mean().round(2))
print('After  scaling — X_train mean:', X_train_sc[:,0].mean().round(4))
print('After  scaling — X_train std: ', X_train_sc[:,0].std().round(4))

## Step 6: Train Models

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)
y_pred_rf = rf.predict(X_test_sc)
acc_rf = accuracy_score(y_test, y_pred_rf)

cv_rf = cross_val_score(rf, X_train_sc, y_train, cv=5)
print(f'Random Forest — Test Accuracy: {acc_rf:.4f} ({acc_rf:.2%})')
print(f'Random Forest — 5-Fold CV:     {cv_rf.mean():.4f} ± {cv_rf.std():.4f}')

In [ ]:
# KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_sc, y_train)
y_pred_knn = knn.predict(X_test_sc)
acc_knn = accuracy_score(y_test, y_pred_knn)

cv_knn = cross_val_score(knn, X_train_sc, y_train, cv=5)
print(f'KNN — Test Accuracy: {acc_knn:.4f} ({acc_knn:.2%})')
print(f'KNN — 5-Fold CV:     {cv_knn.mean():.4f} ± {cv_knn.std():.4f}')

## Step 7: Evaluate Models

In [ ]:
# Model comparison
fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['Random Forest', 'KNN'],
              [acc_rf * 100, acc_knn * 100],
              color=['#1D9E75', '#185FA5'], edgecolor='white', width=0.4)
ax.set_ylim(70, 100); ax.set_ylabel('Accuracy (%)')
ax.set_title('Model Accuracy Comparison', fontsize=13)
for bar, a in zip(bars, [acc_rf, acc_knn]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3, f'{a:.2%}',
            ha='center', va='bottom', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix — Random Forest
fig, ax = plt.subplots(figsize=(14, 12))
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=le.classes_, yticklabels=le.classes_, linewidths=0.3)
ax.set_title(f'Random Forest — Confusion Matrix (Acc: {acc_rf:.2%})', fontsize=13)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix — KNN
fig, ax = plt.subplots(figsize=(14, 12))
cm_knn = confusion_matrix(y_test, y_pred_knn)
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=le.classes_, yticklabels=le.classes_, linewidths=0.3)
ax.set_title(f'KNN — Confusion Matrix (Acc: {acc_knn:.2%})', fontsize=13)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# Classification report — Random Forest
print('Random Forest — Classification Report')
print('=' * 60)
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

In [ ]:
# Classification report — KNN
print('KNN — Classification Report')
print('=' * 60)
print(classification_report(y_test, y_pred_knn, target_names=le.classes_))

## Step 8: Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print('Feature Importances:')
print(importances.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
importances.sort_values().plot(kind='barh', ax=ax, color='#1D9E75', edgecolor='white')
ax.set_title('Feature Importance — Random Forest', fontsize=13)
plt.tight_layout(); plt.show()

## Step 9: Save Models

In [ ]:
import os
os.makedirs('models', exist_ok=True)
joblib.dump(rf,     'models/rf_model.pkl')
joblib.dump(knn,    'models/knn_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(le,     'models/label_encoder.pkl')
print('Models saved to models/')

## Step 10: Test Prediction

In [ ]:
# Example prediction
sample = np.array([[90, 42, 43, 20, 82, 6.5, 200]])  # rice-like conditions
sample_sc = scaler.transform(sample)
pred = rf.predict(sample_sc)
proba = rf.predict_proba(sample_sc)[0]
top3 = np.argsort(proba)[::-1][:3]

print(f'Input: N=90, P=42, K=43, Temp=20°C, Humidity=82%, pH=6.5, Rainfall=200mm')
print(f'\nPredicted crop: {le.inverse_transform(pred)[0].upper()}')
print(f'\nTop 3 predictions:')
for i in top3:
    print(f'  {le.classes_[i]:<15} {proba[i]:.2%}')